# 1. Legacy Canonical Songs Table

**Status: legacy / not part of the current n-gram pipeline.**

The current research pipeline studies exact chord n-gram vocabularies `V_n` and harmonic matrix classes `H_n` using binary `12 x n` pitch-class matrices. That work starts in notebook 2.

This notebook is retained only as earlier exploratory work for type normalization and tensor experiments. Do not run the tensor or scraping sections for the current analysis.

In [1]:
import numpy as np
import pandas as pd

# Legacy Canonical Songs Table

Earlier goals were:

1. Normalize data types.
2. Create `chords_nosections` by stripping section tags from `chords`.
3. Create and save 3-channel tensors.

For the current project direction, only section stripping and metadata normalization remain relevant. The tensor artifacts are not used by notebooks 2-4.

In [2]:
DATAPATH = '../data/raw/'
FILENAME = 'chordonomicon_v2.csv'

In [3]:
df = pd.read_csv(DATAPATH + FILENAME,low_memory=False)
df.head()

,id,chords,release_date,genres,decade,rock_genre,artist_id,main_genre,spotify_song_id,spotify_artist_id
0,1,<intro_1> C <verse_1> F C E7 Amin C F C G7 C F...,NaN,'classic country pop',NaN,NaN,artist_1,pop,NaN,4AIEGdwDzPELXYgM5JaEY5
1,2,<intro_1> E D A/Cs E D A/Cs <verse_1> E D A/Cs...,2003-01-01,'alternative metal' 'alternative rock' 'nu met...,2000.0,pop rock,artist_2,metal,2ffJZ2r8HxI5DHcmf3BO6c,694QW15WkebjcrWgQHzRYF
2,3,<intro_1> Csmin <verse_1> A Csmin A Csmin A Cs...,2003-01-01,'alternative metal' 'canadian rock' 'funk meta...,2000.0,canadian rock,artist_3,metal,5KiY8SZEnvCPyIEkFGRR3y,0niJkG4tKkne3zwr7I8n9n
3,4,<intro_1> D Dmaj7 D Dmaj7 <verse_1> Emin A D G...,2022-09-23,NaN,2020.0,NaN,artist_4,NaN,01TtAcUqyLCRBZq4ZZiQWS,17BfKBemmMGO5ZAK25wraW
4,5,<intro_1> C <verse_1> G C G C <chorus_1> F Dmi...,2023-02-10,'modern country pop',2020.0,NaN,artist_5,pop,3zUecdrWC3IqrNSjhnoF3G,4GGfAshSkqoxpZdoaHm7ky


In [4]:
df = df.set_index('id')

## Tensor Construction
From a chord (e.g. Cmaj7/D = D-C-E-G-B), we can consider the binary 12-vector by one hot encoding each note way relative to C modulo octaves: 
$$  \text{Cmaj7/D} \mapsto \begin{bmatrix} 1 & 0 & 1 & 0 & 1 & 0 & 0 & 1 & 0 &0 &0 & 1 \end{bmatrix} ^T. $$
We can decompose  this chord as a tuple given by (**core**,**extensions**,**bass**), where **core** is the core triad/diad of a chord (e.g. Cmaj), the **extensions** are the upper extensions of the chord (e.g. B, the major 7 of C), and **bass** being the lowest note (e.g. D). For the previous example, this gives 
$$ \text{Cmaj7/D} \mapsto \text{(core,extensions,bass)}=\text{(C-E-G, B, D)}.$$
 

If we consider a song $s$ as a list of (untagged) chords, then one can consider the $3\times 12 \times L$ tensor defined by the above process, where 3 corresponds to the channel dimension (core,extensions,bass), 12 corresponds to out pitch classes, and $L$ is the number of chords in $s$.

In [5]:
from utils import normalize_chords as nc 

df['chords_nosections'] = df['chords'].map(nc.remove_sections)
df.head()

,chords,release_date,genres,decade,rock_genre,artist_id,main_genre,spotify_song_id,spotify_artist_id,chords_nosections
id,,,,,,,,,,
1,<intro_1> C <verse_1> F C E7 Amin C F C G7 C F...,NaN,'classic country pop',NaN,NaN,artist_1,pop,NaN,4AIEGdwDzPELXYgM5JaEY5,C F C E7 Amin C F C G7 C F C E7 Amin C F G7 C ...
2,<intro_1> E D A/Cs E D A/Cs <verse_1> E D A/Cs...,2003-01-01,'alternative metal' 'alternative rock' 'nu met...,2000.0,pop rock,artist_2,metal,2ffJZ2r8HxI5DHcmf3BO6c,694QW15WkebjcrWgQHzRYF,E D A/Cs E D A/Cs E D A/Cs E D A/Cs E D A/Cs E...
3,<intro_1> Csmin <verse_1> A Csmin A Csmin A Cs...,2003-01-01,'alternative metal' 'canadian rock' 'funk meta...,2000.0,canadian rock,artist_3,metal,5KiY8SZEnvCPyIEkFGRR3y,0niJkG4tKkne3zwr7I8n9n,Csmin A Csmin A Csmin A Csmin A B Csmin A Fsmi...
4,<intro_1> D Dmaj7 D Dmaj7 <verse_1> Emin A D G...,2022-09-23,NaN,2020.0,NaN,artist_4,NaN,01TtAcUqyLCRBZq4ZZiQWS,17BfKBemmMGO5ZAK25wraW,D Dmaj7 D Dmaj7 Emin A D G Emin A D G Emin A D...
5,<intro_1> C <verse_1> G C G C <chorus_1> F Dmi...,2023-02-10,'modern country pop',2020.0,NaN,artist_5,pop,3zUecdrWC3IqrNSjhnoF3G,4GGfAshSkqoxpZdoaHm7ky,C G C G C F Dmin G Dmin G C G C F Dmin G Dmin ...


In [6]:
# Tensor export is legacy and disabled by default.
# Notebooks 2-6 do not use these rank-3 tensor artifacts.
RUN_TENSOR_EXPORT = False

if RUN_TENSOR_EXPORT:
    from pathlib import Path
    import torch
    from tqdm import tqdm

    from utils import chords_to_tensor as ctt

    out_dir = Path("../data/processed/tensors")
    out_dir.mkdir(parents=True, exist_ok=True)

    paths = []
    for song_id, chord_str in tqdm(
        zip(df.index, df["chords_nosections"]),
        total=len(df),
    ):
        path = out_dir / f"{song_id}.pt"

        if not path.exists():
            x = ctt.chord_string_to_rank3_tensor(chord_str)
            torch.save(x, path)

        paths.append(str(path))

    df["tensor_path"] = paths
else:
    print("Skipped legacy tensor export. Current pipeline starts in notebook 2.")


Skipped legacy tensor export. Current pipeline starts in notebook 2.


## Data Type Normalization

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 679807 entries, 1 to 679807
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   chords             679807 non-null  object 
 1   release_date       422181 non-null  object 
 2   genres             429753 non-null  object 
 3   decade             422181 non-null  float64
 4   rock_genre         145218 non-null  object 
 5   artist_id          510986 non-null  object 
 6   main_genre         352111 non-null  object 
 7   spotify_song_id    440284 non-null  object 
 8   spotify_artist_id  510986 non-null  object 
 9   chords_nosections  679807 non-null  object 
dtypes: float64(1), object(9)
memory usage: 57.1+ MB


In [8]:
df["release_date"] = df["release_date"].map(
    lambda x: pd.to_datetime(x, errors="coerce")
)

df["decade"] = df["decade"].astype("Int64")

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 679807 entries, 1 to 679807
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   chords             679807 non-null  object        
 1   release_date       422181 non-null  datetime64[ns]
 2   genres             429753 non-null  object        
 3   decade             422181 non-null  Int64         
 4   rock_genre         145218 non-null  object        
 5   artist_id          510986 non-null  object        
 6   main_genre         352111 non-null  object        
 7   spotify_song_id    440284 non-null  object        
 8   spotify_artist_id  510986 non-null  object        
 9   chords_nosections  679807 non-null  object        
dtypes: Int64(1), datetime64[ns](1), object(8)
memory usage: 57.7+ MB


In [10]:
df.sample(10)

,chords,release_date,genres,decade,rock_genre,artist_id,main_genre,spotify_song_id,spotify_artist_id,chords_nosections
id,,,,,,,,,,
221604,<verse_1> Emin D C D Emin D C D C D Emin C B7 ...,NaT,'viral pop',<NA>,NaN,artist_36731,pop,NaN,4zrB9WV6Z0iKKnGb1Pj0P2,Emin D C D Emin D C D C D Emin C B7 Emin D C G...
411538,Emin D Emin D G Amin Emin D Emin D G Amin Emin...,2018-11-16,'girl group' 'pop' 'talent show' 'uk pop',2010,NaN,artist_41870,pop,3QKI38J8BZcTTF1HusbUKh,3e7awlrlDSwF3iM0WBjGMp,Emin D Emin D G Amin Emin D Emin D G Amin Emin...
264738,<intro_1> E/B E/C E/Cs E/C E/B E/A E/B E/C E/C...,NaT,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,E/B E/C E/Cs E/C E/B E/A E/B E/C E/Cs E/C E/B ...
270082,<intro_1> A E Fsmin D <verse_1> A E Fsmin D A ...,2015-08-14,'jawaiian',2010,NaN,artist_59823,NaN,6MX6kCuCWuTZL9YzRWVo8q,7vPGzgDg3qGUY5bWtrO3K4,A E Fsmin D A E Fsmin D A E Fsmin D A E Fsmin ...
622685,<intro_1> E G A D <verse_1> E G A D E <verse_2...,2009-09-15,'canadian americana' 'canadian singer-songwrit...,2000,NaN,artist_65619,alternative,5vPQNVFOlKDSFwsxxh7OIz,6VE2XJCMlwhlF3vtMo50aj,E G A D E G A D E G A D E D A E D A E D A E G ...
92030,<verse_1> E A Fsmin Csmin E <verse_2> A Fsmin ...,2020-08-28,NaN,2010,NaN,artist_32676,NaN,17V9MiWWm3QbwKi2PUPUBK,1TsPGD8cCf3JaSGrC7sLkf,E A Fsmin Csmin E A Fsmin Csmin E A E A E Fsmi...
492046,Dmin Gmin C F E Amin A7 Dmin Gmin C F Amin Bb ...,1998-01-09,NaN,1990,NaN,artist_83721,NaN,7icSo8AIxXsxKgqKVddu3s,6N58FlJZYmyb541ZvpW2Ka,Dmin Gmin C F E Amin A7 Dmin Gmin C F Amin Bb ...
625721,<intro_1> A E D <verse_1> A E/Gs D A E/Gs D Fs...,NaT,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,A E D A E/Gs D A E/Gs D Fsmin E D Bmin Csmin F...
396746,Gsmin Fsadd13 E Eadd13 Gsmin Fsadd13 E Eadd13 ...,2006-04-25,'neo soul' 'r&b' 'urban contemporary',2000,NaN,artist_72198,soul,1e3wsCDsCZHBsdc1QTeXif,4hVcxmC7igpot32EzQf7IR,Gsmin Fsadd13 E Eadd13 Gsmin Fsadd13 E Eadd13 ...


## Optional Spotify Name Scraping Removed From Run Path

This was previously an executable cell that scraped public Spotify pages for song and artist names. It is intentionally no longer executable in this notebook because it is slow, brittle, network-dependent, and unrelated to n-gram frequency construction.

If metadata enrichment becomes important later, build it as a separate cached/API-backed workflow.

## Save dataframe as `songs_master.parquet`

In [11]:
# Save the dataframe

from pathlib import Path

# project root = one level up from notebooks
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

df.to_parquet(DATA_DIR / "songs_master.parquet", index=True)